# 01 — Preprocessing
Chargement et nettoyage des données DECP 2022.

In [1]:
import json
import pandas as pd

In [2]:
with open('../data/decp-2022.json', 'r') as f:
    data = json.load(f)

marches = data['marches']
print(f'{len(marches)} marchés chargés')

8756 marchés chargés


In [3]:
print(json.dumps(marches[0], ensure_ascii=False, indent=2))

{
  "id": "2022-884095-01",
  "acheteur": {
    "id": "21440114300018"
  },
  "nature": "Marché",
  "objet": "Transports collectifs pour la Ville d'Orvault: Lot 1: Circuits sur la commune d\\'Orvault",
  "codeCPV": "60130000",
  "procedure": "Procédure adaptée",
  "lieuExecution": {
    "code": "44700",
    "typeCode": "Code postal"
  },
  "dureeMois": 12.0,
  "dateNotification": "2022-07-29",
  "datePublicationDonnees": "2024-08-30",
  "montant": 46000.0,
  "formePrix": "Unitaire",
  "attributionAvance": "non",
  "offresRecues": 1,
  "marcheInnovant": "non",
  "ccag": "Pas de CCAG",
  "sousTraitanceDeclaree": "non",
  "typeGroupementOperateurs": "Pas de groupement",
  "titulaires": [
    {
      "titulaire": {
        "typeIdentifiant": "SIRET",
        "id": "87080218800058"
      }
    }
  ],
  "considerationsSociales": {
    "considerationSociale": [
      "Pas de considération sociale"
    ]
  },
  "considerationsEnvironnementales": {
    "considerationEnvironnementale": [
      "

In [4]:
rows = []

for m in marches:
    acheteur = m.get('acheteur', {})
    acheteur_id = acheteur.get('id') if isinstance(acheteur, dict) else None
    titulaires = m.get('titulaires', [])
    nb_titulaires = len(titulaires) if isinstance(titulaires, list) else 0

    rows.append({
        'id': m.get('id'),
        'objet': m.get('objet', ''),
        'acheteur_id': acheteur_id,
        'montant': m.get('montant'),
        'dureeMois': m.get('dureeMois'),
        'offresRecues': m.get('offresRecues'),
        'procedure': m.get('procedure'),
        'codeCPV': str(m.get('codeCPV', ''))[:2],
        'nb_titulaires': nb_titulaires,
        'dateNotification': m.get('dateNotification'),
    })

df = pd.DataFrame(rows)
df.head()

,id,objet,acheteur_id,montant,dureeMois,offresRecues,procedure,codeCPV,nb_titulaires,dateNotification
0,2022-884095-01,Transports collectifs pour la Ville d'Orvault:...,21440114300018,46000.0,12.0,1,Procédure adaptée,60,1,2022-07-29
1,2024-0202403100,"Travaux d'éclairage public, de signalisation l...",21680066400015,253530.0,8.0,2,Procédure adaptée,45,1,2024-07-11
2,24-131-00,"LE MOLAY LITTRY - 53, 55, 57, 59, 61 et 63 rou...",78070570300012,113278.0,2.0,1,Procédure adaptée,45,1,2024-08-29
3,22-076,"Réalisation 11 logements collectifs, lot B5-D3...",45220075100025,46678.0,24.0,1,Procédure avec négociation,44,1,2022-11-23
4,22-207,Construction de 2 bâtiments totalisant 44 loge...,45220075100025,59354.0,22.0,1,Procédure adaptée,44,1,2022-12-07


In [5]:
print(df.dtypes)

id                      str
objet                   str
acheteur_id             str
montant             float64
dureeMois            object
offresRecues          int64
procedure               str
codeCPV                 str
nb_titulaires         int64
dateNotification        str
dtype: object


In [6]:
for col in ['montant', 'dureeMois', 'offresRecues']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['dateNotification'] = pd.to_datetime(df['dateNotification'], errors='coerce')

print('Valeurs manquantes :')
print(df.isnull().sum())

Valeurs manquantes :
id                   0
objet                0
acheteur_id         24
montant              0
dureeMois            0
offresRecues         0
procedure            0
codeCPV              0
nb_titulaires        0
dateNotification    24
dtype: int64


In [7]:
for col in ['montant', 'dureeMois', 'offresRecues']:
    df[col] = df[col].fillna(0)

print('Nettoyage terminé')
df.describe()

Nettoyage terminé


,montant,dureeMois,offresRecues,nb_titulaires,dateNotification
count,8.756000e+03,8756.000000,8756.000000,8756.000000,8732
mean,9.705152e+05,21.312357,4.925194,1.148470,2023-04-05 22:56:20.668804
min,0.000000e+00,1.000000,0.000000,0.000000,0002-09-30 00:00:00
25%,3.201575e+04,8.000000,2.000000,1.000000,2024-02-19 00:00:00
50%,8.960450e+04,14.000000,3.000000,1.000000,2024-05-14 00:00:00
75%,2.500000e+05,28.000000,4.000000,1.000000,2024-07-09 00:00:00
max,2.024300e+09,4015.000000,13454.000000,13.000000,2024-10-10 00:00:00
std,2.511960e+07,46.464856,143.835999,0.684744,NaN


In [8]:
df.to_parquet('../data/decp_clean.parquet', index=False)
print('Sauvegardé : data/decp_clean.parquet')

Sauvegardé : data/decp_clean.parquet
